# Draw figures directly from wandb, so convinent!

In [ ]:
import wandb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

api = wandb.Api()

ENTITY = "<WANDB_ENTITY>"
PROJECT_cpt = "plasticity-loss"
PROJECT_sft = "plasticity-step6"

def name_to_id_cpt(name):
    runs = api.runs(
        f"{ENTITY}/{PROJECT_cpt}",
        filters={
            "display_name": name
        },
    )
    return runs[0].id

def name_to_id_sft(name):
    runs = api.runs(
        f"{ENTITY}/{PROJECT_sft}",
        filters={
            "display_name": name
        },
    )
    return runs[0].id

def load_train_curve(run):
    rows = list(
        run.scan_history(
            keys=["epoch", "train/loss"],
            page_size=1000,
        )
    )

    df = pd.DataFrame(rows)

    df = (
        df
        .dropna(subset=["epoch", "train/loss"])
        .sort_values("epoch")
        .reset_index(drop=True)
    )

    return df

In [ ]:
all_runs = list(api.runs(f"{ENTITY}/{PROJECT_sft}"))

In [ ]:
runs_by_model = {}

for r in all_runs:
    job = r.config.get("job", {})

    model = job.get("model_name")
    task = job.get("task")
    ckpt = job.get("checkpoint_label")

    if model is None or task is None or ckpt is None:
        continue

    runs_by_model.setdefault(model, {})
    runs_by_model[model].setdefault(task, {})

    runs_by_model[model][task][ckpt] = r

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FixedLocator, FuncFormatter


# ============================================================
# 0. Choose model
# ============================================================

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"#"allenai/OLMo-1B-hf"#"Qwen/Qwen2.5-0.5B-Instruct"

# Other models:
# MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL = "allenai/OLMo-1B-hf"


# ============================================================
# 1. Basic setup
# ============================================================

tasks = {
    "GSM8K": "gsm8k",
    "MBPP": "mbpp",
    "Dolly-QA": "dolly_qa",
}

checkpoint_order = [
    "base",
    "5M",
    "10M",
    "20M",
    "40M",
    "99M",
]

ckpt_name_map = {
    "base": "base",
    "checkpoint_5M_tokens": "5M",
    "checkpoint_10M_tokens": "10M",
    "checkpoint_20M_tokens": "20M",
    "checkpoint_40M_tokens": "40M",
    "checkpoint_99M_tokens": "99M",
}


# ============================================================
# 2. Select runs for this model
# ============================================================

selected_runs = {}

for task_key, ckpts in runs_by_model[MODEL].items():

    selected_runs[task_key] = {}

    for raw_ckpt_name, run in ckpts.items():

        if raw_ckpt_name not in ckpt_name_map:
            continue

        nice_name = ckpt_name_map[raw_ckpt_name]
        selected_runs[task_key][nice_name] = run


# Sanity check
for task_key in tasks.values():
    print(
        task_key,
        sorted(
            selected_runs[task_key].keys(),
            key=lambda x: checkpoint_order.index(x)
        )
    )


# ============================================================
# 3. W&B loaders
# ============================================================

def load_curve(run, x_key, y_key):

    rows = list(
        run.scan_history(
            keys=[x_key, y_key],
            page_size=1000,
        )
    )

    if len(rows) == 0:
        return pd.DataFrame(columns=["x", "y"])

    df = pd.DataFrame(rows)

    if x_key not in df.columns or y_key not in df.columns:
        return pd.DataFrame(columns=["x", "y"])

    df = (
        df
        .dropna(subset=[x_key, y_key])
        .sort_values(x_key)
        .rename(
            columns={
                x_key: "x",
                y_key: "y",
            }
        )
        [["x", "y"]]
        .reset_index(drop=True)
    )

    return df


def load_train_curve(run):

    return load_curve(
        run,
        x_key="epoch",
        y_key="train/loss",
    )


def load_probe_curve(run):
    """
    Prefer epoch as the x-axis.
    Fall back to sft_global_step if epoch is unavailable.
    """

    df = load_curve(
        run,
        x_key="epoch",
        y_key="probe/loss",
    )

    if len(df) > 0:
        return df, "epoch"

    df = load_curve(
        run,
        x_key="sft_global_step",
        y_key="probe/loss",
    )

    return df, "step"


# ============================================================
# 4. Robust smoothing
#    Used only for GSM8K training curves
# ============================================================

def robust_smooth(
    y,
    window=9,
    n_sigma=3.5,
    ema_alpha=0.22,
    despike=True,
):

    y = pd.Series(
        np.asarray(y, dtype=float)
    ).clip(lower=1e-12)

    # Since the final figure uses a log y-axis,
    # smooth multiplicative variations in log space.
    log_y = np.log(y)

    if despike:

        rolling_median = log_y.rolling(
            window=window,
            center=True,
            min_periods=1,
        ).median()

        deviation = (
            log_y - rolling_median
        ).abs()

        mad = deviation.rolling(
            window=window,
            center=True,
            min_periods=1,
        ).median()

        robust_sigma = 1.4826 * mad

        is_outlier = (
            deviation > n_sigma * robust_sigma
        )

        # Do not classify points when local MAD is zero.
        is_outlier &= robust_sigma > 1e-8

        clean_log_y = log_y.mask(is_outlier)

        clean_log_y = clean_log_y.interpolate(
            method="linear",
            limit_direction="both",
        )

    else:
        clean_log_y = log_y

    smooth_log_y = clean_log_y.ewm(
        alpha=ema_alpha,
        adjust=False,
    ).mean()

    return np.exp(
        smooth_log_y.to_numpy()
    )


# ============================================================
# 5. Clean log-scale Y ticks
# ============================================================

def decimal_formatter(x, pos):

    if x >= 10:
        return f"{x:.0f}"

    if x >= 1:
        return f"{x:.1f}".rstrip("0").rstrip(".")

    if x >= 0.1:
        return f"{x:.2f}".rstrip("0").rstrip(".")

    return f"{x:.3f}".rstrip("0").rstrip(".")


def set_clean_log_yticks(ax, min_ticks=3, max_ticks=5):
    """
    Select roughly 3--5 visually clean decimal tick values
    while preserving the logarithmic y-axis.
    """

    ymin, ymax = ax.get_ylim()

    # Nice mantissas over several decades.
    mantissas = np.array([
        1.0, 1.2, 1.5, 2.0, 2.5,
        3.0, 4.0, 5.0, 6.0, 8.0
    ])

    emin = int(np.floor(np.log10(ymin))) - 1
    emax = int(np.ceil(np.log10(ymax))) + 1

    candidates = []

    for exponent in range(emin, emax + 1):
        candidates.extend(
            mantissas * (10.0 ** exponent)
        )

    candidates = np.array(sorted(candidates))

    ticks = candidates[
        (candidates >= ymin) &
        (candidates <= ymax)
    ]

    # Too many -> sample evenly in log order
    if len(ticks) > max_ticks:

        idx = np.linspace(
            0,
            len(ticks) - 1,
            max_ticks
        ).round().astype(int)

        ticks = ticks[idx]

    # In an unusually narrow range, construct geometric ticks.
    if len(ticks) < min_ticks:

        ticks = np.geomspace(
            ymin,
            ymax,
            min_ticks
        )

    ax.yaxis.set_major_locator(
        FixedLocator(ticks)
    )

    ax.yaxis.set_major_formatter(
        FuncFormatter(decimal_formatter)
    )

    # Minor ticks remain, but no labels.
    ax.tick_params(
        axis="y",
        which="minor",
        labelleft=False,
    )


# ============================================================
# 6. Download all curves
# ============================================================

data = {}

probe_x_type = {}

for task_name, task_key in tasks.items():

    data[task_name] = {
        "train": {},
        "probe": {},
    }

    for ckpt in checkpoint_order:

        run = selected_runs[task_key][ckpt]

        print(
            f"Loading {task_name:8s} | {ckpt:4s}"
        )

        # Training loss
        data[task_name]["train"][ckpt] = (
            load_train_curve(run)
        )

        # Probe / validation loss
        probe_df, x_type = load_probe_curve(run)

        data[task_name]["probe"][ckpt] = probe_df

        probe_x_type[
            (task_name, ckpt)
        ] = x_type


# ============================================================
# 7. Load the same curves for all three CPT models
# ============================================================

MODEL_SPECS = [
    ("Qwen/Qwen2.5-0.5B-Instruct", "q0p5b"),
    ("Qwen/Qwen2.5-1.5B-Instruct", "q1p5b"),
    ("allenai/OLMo-1B-hf", "olmo"),
]

data_by_model = {MODEL: data}
probe_x_type_by_model = {MODEL: probe_x_type}

for model_name, _ in MODEL_SPECS:
    if model_name == MODEL:
        continue

    if model_name not in runs_by_model:
        raise KeyError(f"No SFT runs found for {model_name}")

    model_runs = {}
    for task_key, ckpts in runs_by_model[model_name].items():
        model_runs[task_key] = {}
        for raw_ckpt_name, run in ckpts.items():
            if raw_ckpt_name in ckpt_name_map:
                model_runs[task_key][ckpt_name_map[raw_ckpt_name]] = run

    missing = {
        task_key: [
            checkpoint
            for checkpoint in checkpoint_order
            if checkpoint not in model_runs.get(task_key, {})
        ]
        for task_key in tasks.values()
    }
    missing = {task_key: values for task_key, values in missing.items() if values}
    if missing:
        raise RuntimeError(f"Missing runs for {model_name}: {missing}")

    model_data = {}
    model_probe_x_type = {}
    for task_name, task_key in tasks.items():
        model_data[task_name] = {"train": {}, "probe": {}}
        for checkpoint in checkpoint_order:
            run = model_runs[task_key][checkpoint]
            print(f"Loading {model_name} | {task_name} | {checkpoint}")
            model_data[task_name]["train"][checkpoint] = load_train_curve(run)
            probe_df, x_type = load_probe_curve(run)
            model_data[task_name]["probe"][checkpoint] = probe_df
            model_probe_x_type[(task_name, checkpoint)] = x_type

    data_by_model[model_name] = model_data
    probe_x_type_by_model[model_name] = model_probe_x_type


In [ ]:
# ============================================================
# 7. Colors
# ============================================================

cmap = plt.get_cmap("viridis")
cpt_ckpts = ["5M", "10M", "20M", "40M", "99M"]
cpt_colors = cmap(np.linspace(0.15, 0.85, len(cpt_ckpts)))
colors = {"base": "black"}
colors.update(dict(zip(cpt_ckpts, cpt_colors)))


def plot_panel(ax, curves, task_name, curve_type):
    for ckpt in checkpoint_order:
        df = curves[ckpt]
        if len(df) == 0:
            print(f"Warning: empty curve {task_name} / {curve_type} / {ckpt}")
            continue

        x = df["x"].to_numpy(dtype=float)
        y = df["y"].to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)
        x, y = x[valid], y[valid]

        if task_name == "GSM8K" and curve_type == "train":
            y = robust_smooth(y, window=9, n_sigma=3.5, ema_alpha=0.22, despike=True)

        ax.plot(x, y, color=colors[ckpt], linewidth=1.8, label=ckpt)

    ax.set_xscale("log")
    ax.set_yscale("log")
    set_clean_log_yticks(ax, min_ticks=3, max_ticks=4)
    ax.grid(True, which="major", alpha=0.15, linewidth=0.6)
    ax.grid(True, which="minor", alpha=0.035, linewidth=0.4)
    ax.tick_params(axis="both", which="major", labelsize=9)


legend_handles = [
    Line2D([0], [0], color=colors[ckpt], linewidth=2.2)
    for ckpt in checkpoint_order
]
legend_labels = ["Base", "5M", "10M", "20M", "40M", "99M"]


for model_name, file_tag in MODEL_SPECS:
    model_data = data_by_model[model_name]
    model_probe_x_type = probe_x_type_by_model[model_name]

    fig, axes = plt.subplots(2, 3, figsize=(11.5, 5.7))

    for col, task_name in enumerate(tasks):
        top_ax = axes[0, col]
        plot_panel(
            top_ax,
            model_data[task_name]["train"],
            task_name=task_name,
            curve_type="train",
        )
        top_ax.set_title(task_name, fontsize=13, pad=8)
        top_ax.set_xlabel("SFT Epoch", fontsize=10)

        bottom_ax = axes[1, col]
        plot_panel(
            bottom_ax,
            model_data[task_name]["probe"],
            task_name=task_name,
            curve_type="probe",
        )

    axes[0, 0].set_ylabel("Training Loss", fontsize=11)
    axes[1, 0].set_ylabel("Validation Loss", fontsize=11)

    all_probe_types = set(model_probe_x_type.values())
    bottom_xlabel = (
        "Continue SFT Epoch"
        if all_probe_types == {"epoch"}
        else "Continue SFT Step"
    )
    for ax in axes[1, :]:
        ax.set_xlabel(bottom_xlabel, fontsize=10)

    fig.legend(
        handles=legend_handles,
        labels=legend_labels,
        loc="upper center",
        bbox_to_anchor=(0.5, 1.015),
        ncol=6,
        frameon=False,
        fontsize=10,
        handlelength=2.2,
        columnspacing=1.7,
    )
    fig.subplots_adjust(
        left=0.075,
        right=0.995,
        bottom=0.09,
        top=0.88,
        wspace=0.16,
        hspace=0.30,
    )

    output_path = f"./figures/cpt_plasticity_curves_{file_tag}.pdf"
    fig.savefig(output_path, bbox_inches="tight", dpi=300)
    plt.show()
    print(f"Saved: {output_path}")

## Main context, reset, and reset last 4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


# ============================================================
# 0. Choose model
# ============================================================

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"

# Other choices:
# MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
# MODEL = "allenai/OLMo-1B-hf"


# ============================================================
# 1. Find W&B run by display name + model
# ============================================================

def get_run_by_name_and_model(
    run_name,
    model_name,
    project=PROJECT_sft,
):
    """
    Find a W&B run using both:
      1. display_name
      2. config['job']['model_name']

    If multiple matching reruns exist, use the newest one.
    """

    matches = list(
        api.runs(
            f"{ENTITY}/{project}",
            filters={
                "display_name": run_name
            },
        )
    )

    # Filter by model
    matches = [
        r for r in matches
        if r.config.get("job", {}).get("model_name") == model_name
    ]

    if len(matches) == 0:
        raise ValueError(
            f"Cannot find run:\n"
            f"  name    = {run_name}\n"
            f"  model   = {model_name}\n"
            f"  project = {project}"
        )

    # If there are duplicated reruns, use the newest
    if len(matches) > 1:
        matches = sorted(
            matches,
            key=lambda r: r.created_at,
            reverse=True,
        )

        print(
            f"[Warning] Found {len(matches)} runs for "
            f"{run_name} / {model_name}. "
            f"Using newest: {matches[0].id}"
        )

    return matches[0]


# ============================================================
# 2. Load one curve
# ============================================================

def load_curve(run, x_key, y_key):

    rows = list(
        run.scan_history(
            keys=[x_key, y_key],
            page_size=1000,
        )
    )

    df = pd.DataFrame(rows)

    if (
        len(df) == 0
        or x_key not in df.columns
        or y_key not in df.columns
    ):
        return pd.DataFrame(
            columns=["x", "y"]
        )

    df = (
        df
        .dropna(subset=[x_key, y_key])
        .sort_values(x_key)
        .rename(
            columns={
                x_key: "x",
                y_key: "y",
            }
        )
        [["x", "y"]]
        .reset_index(drop=True)
    )

    return df


# ============================================================
# 3. Plot reset comparison
# ============================================================

def plot_reset_single(
    task_title,
    run_names,
    model_name=MODEL,
    project=PROJECT_sft,
    metric="train",
    colors=None,
    save_path=None,
    figsize=(4.8, 3.6),
    x_lim=None,
    y_lim=None,
    x_ticks=None,
    y_ticks=None,
    legend=True,
):

    if colors is None:
        colors = [
            "black",      # Base
            "#67B7DC",    # No reset
            "#F28E2B",    # Reset readout
            "#59A14F",    # Reset last 4
        ]

    curve_specs = [
        ("base", "Base", colors[0]),
        ("no_reset", "No reset", colors[1]),
        ("reset_readout", "Reset readout", colors[2]),
        ("reset_last4", "Reset last 4", colors[3]),
    ]


    # ========================================================
    # Metric
    # ========================================================

    if metric == "train":

        x_key = "epoch"
        y_key = "train/loss"

        x_label = "Continue SFT Epoch"
        y_label = "Training Loss"

    elif metric == "probe":

        x_key = "epoch"
        y_key = "probe/loss"

        x_label = "Continue SFT Epoch"
        y_label = "Validation Loss"

    else:

        raise ValueError(
            "metric must be 'train' or 'probe'"
        )


    # ========================================================
    # Load all four curves
    # ========================================================

    curves = {}

    for key, label, _ in curve_specs:

        run_name = run_names[key]

        run = get_run_by_name_and_model(
            run_name=run_name,
            model_name=model_name,
            project=project,
        )

        print(
            f"{label:14s} | "
            f"{run.name:50s} | "
            f"{run.id}"
        )

        curves[key] = load_curve(
            run,
            x_key=x_key,
            y_key=y_key,
        )


    # ========================================================
    # Plot
    # ========================================================

    fig, ax = plt.subplots(
        figsize=figsize
    )

    for key, label, color in curve_specs:

        df = curves[key]

        if len(df) == 0:
            print(
                f"[Warning] Empty curve: {key}"
            )
            continue

        x = df["x"].to_numpy(
            dtype=float
        )

        y = df["y"].to_numpy(
            dtype=float
        )

        mask = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x > 0)
            & (y > 0)
        )

        x = x[mask]
        y = y[mask]

        ax.plot(
            x,
            y,
            label=label,
            color=color,
            linewidth=2.1,
        )


    # ========================================================
    # Log scales
    # ========================================================

    ax.set_xscale("log")
    ax.set_yscale("log")


    # ========================================================
    # Limits
    # ========================================================

    if x_lim is not None:
        ax.set_xlim(*x_lim)

    if y_lim is not None:
        ax.set_ylim(*y_lim)


    # ========================================================
    # X ticks
    # ========================================================

    if x_ticks is not None:

        ax.set_xticks(x_ticks)

        ax.xaxis.set_major_formatter(
            mticker.FuncFormatter(
                lambda x, pos: f"{x:g}"
            )
        )

    else:

        ax.xaxis.set_major_locator(
            mticker.LogLocator(
                base=10.0,
                numticks=4,
            )
        )

        ax.xaxis.set_major_formatter(
            mticker.LogFormatterMathtext(
                base=10.0
            )
        )

    ax.xaxis.set_minor_formatter(
        mticker.NullFormatter()
    )


    # ========================================================
    # Y ticks
    # ========================================================

    if y_ticks is not None:

        ax.set_yticks(y_ticks)

        ax.yaxis.set_major_formatter(
            mticker.FuncFormatter(
                lambda y, pos: f"{y:g}"
            )
        )

    else:

        ax.yaxis.set_major_locator(
            mticker.LogLocator(
                base=10.0,
                numticks=5,
            )
        )

        ax.yaxis.set_major_formatter(
            mticker.LogFormatterMathtext(
                base=10.0
            )
        )

    ax.yaxis.set_minor_formatter(
        mticker.NullFormatter()
    )


    # ========================================================
    # Labels
    # ========================================================
    if task_title is not None:
        ax.set_title(
            task_title,
            fontsize=15,
            pad=8,
        )

    ax.set_xlabel(
        x_label,
        fontsize=12,
    )

    ax.set_ylabel(
        y_label,
        fontsize=12,
    )

    ax.tick_params(
        axis="both",
        labelsize=10,
    )


    # ========================================================
    # Grid
    # ========================================================

    ax.grid(
        True,
        which="major",
        alpha=0.18,
        linewidth=0.6,
    )

    ax.grid(
        True,
        which="minor",
        alpha=0.04,
        linewidth=0.4,
    )


    # ========================================================
    # Legend
    # ========================================================

    if legend:

        ax.legend(
            frameon=False,
            fontsize=9.5,
            loc="best",
        )


    plt.tight_layout()

    if save_path is not None:

        plt.savefig(
            save_path,
            bbox_inches="tight",
        )

    plt.show()

In [ ]:
run_names_mbpp = {
    "base": "mbpp_base",
    "no_reset": "mbpp_checkpoint_40M_tokens",
    "reset_readout": "mbpp_checkpoint_40M_tokens_base_embedding_readout_tied_trainable",
    "reset_last4": "mbpp_checkpoint_40M_tokens_base_embedding_readout_tied_last4_trainable",
}

In [ ]:
plot_reset_single(
    task_title=None,
    run_names=run_names_mbpp,
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    project=PROJECT_sft,
    metric="train",

    x_lim=(0.08, 5.2),
    y_lim=(0.2, 2.5),

    x_ticks=[0.1, 0.5, 1, 2, 5],
    y_ticks=[0.3, 0.5, 1, 2],

    save_path="./figures/mbpp_reset_train.pdf",
)

In [ ]:
plot_reset_single(
    task_title=None,
    run_names=run_names_mbpp,
    model_name="Qwen/Qwen2.5-0.5B-Instruct",
    project=PROJECT_sft,
    metric="probe",

    x_lim=(0.08, 5.2),

    save_path="./figures/mbpp_reset_valid.pdf",
)

## Draw R

In [ ]:
import wandb
import pandas as pd
import numpy as np

api = wandb.Api()

ENTITY = "<WANDB_ENTITY>"
PROJECT_cpt = "plasticity-loss"



RUN_NAMES = {
    "Qwen2.5-0.5B-Instruct":
        "cpt_plasticity_qwen25_0p5b_lr1e4_wd0p1_readout_ckpts99m_20260813",

    "Qwen2.5-1.5B-Instruct":
        "cpt_plasticity_qwen25_1p5b_default_wd0p1_readout_ckpts99m_20260813",

    "OLMo-1B":
        "cpt_plasticity_olmo1b_default_wd0p1_readout_ckpts99m_olmodata_20260813",
}



def get_run_by_name(run_name):

    matches = list(
        api.runs(
            f"{ENTITY}/{PROJECT_cpt}",
            filters={"display_name": run_name},
        )
    )

    if len(matches) == 0:
        raise ValueError(
            f"Cannot find run with display name:\n{run_name}"
        )

    if len(matches) > 1:
        print(
            f"Warning: found {len(matches)} runs named "
            f"{run_name}. Using the first one."
        )

    return matches[0]



RUNS = {}

for model_name, run_name in RUN_NAMES.items():

    run = get_run_by_name(run_name)

    RUNS[model_name] = run

    print(
        f"{model_name:25s} | "
        f"id={run.id} | "
        f"state={run.state}"
    )

In [ ]:
DATASETS = [
    "gsm8k",
    "mbpp",
    "dolly_qa",
    "bio",
]

R_KEYS = {
    "gsm8k": "plasticity/gsm8k/R_mean",
    "mbpp": "plasticity/mbpp/R_mean",
    "dolly_qa": "plasticity/dolly_qa/R_mean",
    "bio": "plasticity/bio/R_mean",
}

LABELS = {
    "gsm8k": "GSM8K",
    "mbpp": "MBPP",
    "dolly_qa": "Dolly-QA",
    "bio": "PubMed (ID)",
}
X_KEY = "cumulative_training_tokens"

In [ ]:
def load_single_R_curve(run, dataset, x_key=X_KEY):

    y_key = R_KEYS[dataset]

    rows = list(
        run.scan_history(
            keys=[x_key, y_key],
            page_size=1000,
        )
    )

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError(
            f"No history found for {run.name}: {y_key}"
        )

    if x_key not in df.columns:
        raise KeyError(
            f"{x_key} not found in run {run.name}"
        )

    if y_key not in df.columns:
        raise KeyError(
            f"{y_key} not found in run {run.name}"
        )

    df = (
        df
        .dropna(subset=[x_key, y_key])
        .sort_values(x_key)
        .rename(
            columns={
                x_key: "x",
                y_key: "R",
            }
        )
        [["x", "R"]]
        .reset_index(drop=True)
    )

    return df

In [ ]:
X_KEY = "cumulative_training_tokens"

R_KEYS = {
    "gsm8k": "plasticity/gsm8k/R_mean",
    "mbpp": "plasticity/mbpp/R_mean",
    "dolly_qa": "plasticity/dolly_qa/R_mean",
    "bio": "plasticity/bio/R_mean",
}

DATASETS = [
    "gsm8k",
    "mbpp",
    "dolly_qa",
    "bio",
]


all_data = {}

for model_name, run in RUNS.items():

    print(f"\nLoading {model_name}")

    # One API call per model
    raw_df = run.history(
        keys=[
            X_KEY,
            *R_KEYS.values(),
        ],
        samples=10000,
        pandas=True,
    )

    print(f"  raw shape: {raw_df.shape}")

    all_data[model_name] = {}

    for dataset, y_key in R_KEYS.items():

        df = (
            raw_df[[X_KEY, y_key]]
            .dropna()
            .sort_values(X_KEY)
            .rename(columns={
                X_KEY: "x",
                y_key: "R",
            })
            .reset_index(drop=True)
        )

        all_data[model_name][dataset] = df

        print(
            f"  {dataset:8s}: "
            f"{len(df):3d} points | "
            f"x=[{df['x'].min()/1e6:.1f}M, "
            f"{df['x'].max()/1e6:.1f}M] | "
            f"R=[{df['R'].min():.3g}, "
            f"{df['R'].max():.3g}]"
        )

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, FixedLocator, FixedFormatter


# ============================================================
# 1. Colors
# ============================================================

COLORS = {
    "gsm8k": "#1f77b4",      # blue
    "mbpp": "#ff7f0e",       # orange
    "dolly_qa": "#2ca02c",   # green
    "bio": "#9467bd",        # purple
}

LABELS = {
    "gsm8k": "GSM8K",
    "mbpp": "MBPP",
    "dolly_qa": "Dolly-QA",
    "bio": "PubMed (ID)",
}

DATASETS = [
    "gsm8k",
    "mbpp",
    "dolly_qa",
    "bio",
]

MODEL_ORDER = [
    "Qwen2.5-0.5B-Instruct",
    "Qwen2.5-1.5B-Instruct",
    "OLMo-1B",
]


# ============================================================
# 2. Manually specify clean log-scale y ticks
#
# Edit these after seeing the final figure.
# ============================================================

YTICKS = {
    "Qwen2.5-0.5B-Instruct": [
        0.2, 0.3, 0.5, 1, 2, 5
    ],

    "Qwen2.5-1.5B-Instruct": [
        1, 2, 3, 5, 10, 20
    ],

    "OLMo-1B": [
        4, 5, 7, 10, 20
    ],
}

# ============================================================
# 3. Smoothing
# ============================================================

def smooth_curve(y, alpha=0.12):
    """
    EMA smoothing.
    Smaller alpha = stronger smoothing.
    """
    return (
        pd.Series(np.asarray(y, dtype=float))
        .ewm(alpha=alpha, adjust=False)
        .mean()
        .to_numpy()
    )


# ============================================================
# 4. Axis formatters
# ============================================================

def million_tokens(x, pos):
    """
    0 -> 0
    20,000,000 -> 20M
    """
    if abs(x) < 1:
        return "0"
    return f"{x / 1e6:g}M"


from matplotlib.ticker import NullLocator, NullFormatter

def set_clean_log_yticks(ax, ticks):
    """
    Logarithmic y-axis with manually controlled tick positions and labels.
    Call this AFTER all plotting and AFTER ax.set_yscale("log").
    """

    ax.set_yscale("log")

    # Explicit major ticks
    ax.set_yticks(ticks)

    # Explicit human-readable labels
    ax.set_yticklabels(
        [f"{t:g}" for t in ticks]
    )

    # Completely remove automatic minor ticks / labels
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_formatter(NullFormatter())

# ============================================================
# 5. Plot
# ============================================================

fig, axes = plt.subplots(
    1,
    3,
    figsize=(15, 3.6),
    sharex=True,
    sharey=False,
)


for ax, model_name in zip(axes, MODEL_ORDER):

    for dataset in DATASETS:

        df = all_data[model_name][dataset]

        x = df["x"].to_numpy(dtype=float)
        y = df["R"].to_numpy(dtype=float)

        mask = (
            np.isfinite(x)
            & np.isfinite(y)
            & (x >= 0)
            & (y > 0)
        )

        x = x[mask]
        y = y[mask]

        # ----------------------------------------------------
        # Raw curve: faint
        # ----------------------------------------------------

        ax.plot(
            x,
            y,
            color=COLORS[dataset],
            linewidth=0.8,
            alpha=0.16,
        )

        # ----------------------------------------------------
        # Smoothed curve: main trajectory
        # ----------------------------------------------------

        y_smooth = smooth_curve(
            y,
            alpha=0.12,
        )

        ax.plot(
            x,
            y_smooth,
            color=COLORS[dataset],
            linewidth=2.0,
            alpha=0.95,
        )


    # ========================================================
    # Panel formatting
    # ========================================================

    ax.set_title(
        model_name,
        fontsize=12,
        pad=8,
    )

    # clean log-scale ticks
    set_clean_log_yticks(
        ax,
        YTICKS[model_name]
    )

    # Optional: explicitly constrain y-range to tick range.
    # Uncomment if you want identical visual bounds.
    #
    # ax.set_ylim(
    #     min(YTICKS[model_name]),
    #     max(YTICKS[model_name]),
    # )

    # x axis
    ax.xaxis.set_major_formatter(
        FuncFormatter(million_tokens)
    )

    ax.set_xlabel(
        "Long trained Tokens",
        fontsize=11,
    )

    # grid
    ax.grid(
        True,
        which="major",
        alpha=0.18,
        linewidth=0.6,
    )

    # I usually keep minor log grid extremely faint
    ax.grid(
        True,
        which="minor",
        alpha=0.04,
        linewidth=0.35,
    )

    ax.tick_params(
        axis="both",
        labelsize=9,
    )


# ============================================================
# 6. Shared y label
# ============================================================

axes[0].set_ylabel(
    r"$R_{\mathcal{D}}$ (log scale)",
    fontsize=12,
)


# ============================================================
# Legend: only on the first panel
# ============================================================

legend_handles = [
    Line2D(
        [0],
        [0],
        color=COLORS[d],
        linewidth=2.3,
        label=LABELS[d],
    )
    for d in DATASETS
]

axes[0].legend(
    handles=legend_handles,
    loc="upper right",   # "lower left" / "upper left"
    frameon=False,
    fontsize=9,
    ncol=1,
)

# ============================================================
# 8. Layout and save
# ============================================================

plt.tight_layout()
plt.savefig(
    "./figures/cpt_R_mean_three_models.pdf",
    bbox_inches="tight",
)

plt.show()

In [ ]:
# CPT-300M R_D trajectories by model and optimization setting.
# Local CSVs only. Every panel shows EMA-smoothed task-format plasticity probes.

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter

ROOT = Path("outputs/checkpoints")
OUT = Path("./figures")
OUT.mkdir(parents=True, exist_ok=True)

MODELS = [
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B",
    "allenai/OLMo-1B-hf",
    "meta-llama/Llama-3.2-3B-Instruct",
]
MODEL_LABELS = dict(zip(MODELS, [
    "Qwen2.5-0.5B-Instruct", "Qwen2.5-1.5B",
    "OLMo-1B", "Llama-3.2-3B-Instruct",
]))
PROBES = ["pubmed", "math", "qa", "code", "multilingual", "legal"]
PROBE_LABELS = dict(zip(PROBES, [
    "PubMed", "Math", "QA", "Code", "Multilingual", "Legal",
]))
COLORS = dict(zip(PROBES, plt.get_cmap("tab10").colors[:6]))


def config_at(path):
    return dict(
        line.split("=", 1) for line in path.read_text().splitlines()
        if "=" in line
    )


def setting_at(config):
    return f"LR={config['LEARNING_RATE']}"


runs = []
for command in ROOT.glob("cpt_hybrid300m_*/*/run_command.txt"):
    config = config_at(command)
    model, output = config.get("MODEL_NAME"), command.parent
    if model not in MODELS or not (output / "cpt_plasticity_metrics.csv").exists():
        continue
    runs.append({
        "model": model,
        "setting": setting_at(config),
        "lr": float(config["LEARNING_RATE"]),
        "scheduler": config["LR_SCHEDULER_TYPE"],
        "task_csv": output / "cpt_plasticity_metrics.csv",
        "fixed_csv": output / "cpt_fixed_probe_metrics.csv",
        "run": output.parent.name,
    })

runs = pd.DataFrame(runs)
runs["model"] = pd.Categorical(runs["model"], MODELS, ordered=True)
runs = runs.sort_values(
    ["model", "lr", "scheduler"], ascending=[True, False, True]
).reset_index(drop=True)
counts = runs.groupby("model", observed=False).size()
if not (counts == 3).all():
    raise RuntimeError(f"Expected three completed settings per model:\n{counts}")
display(runs[["model", "setting", "run"]])

tasks = []
for run in runs.to_dict("records"):
    frame = pd.read_csv(run["task_csv"])
    frame = frame.loc[
        frame.task.isin(PROBES),
        ["cumulative_training_tokens", "task", "R_mean"],
    ].copy()
    frame["model"], frame["setting"] = run["model"], run["setting"]
    tasks.append(frame)


tasks = pd.concat(tasks, ignore_index=True)


EMA_ALPHA = 0.8
Y_AXIS = {
    "Qwen/Qwen2.5-0.5B-Instruct": ((0.18, 6.0), [0.2, 0.3, 0.5, 1, 2, 5]),
    "Qwen/Qwen2.5-1.5B": ((1.0, 12.0), [1, 2, 3, 5, 10]),
    "allenai/OLMo-1B-hf": ((4.0, 40.0), [4, 5, 7, 10, 20, 30, 40]),
    "meta-llama/Llama-3.2-3B-Instruct": ((0.8, 25.0), [0.8, 1, 2, 3, 5, 10, 20]),
}


def millions(value, _):
    return f"{value / 1e6:g}M"


def ema(values, alpha=EMA_ALPHA):
    return pd.Series(values, dtype=float).ewm(alpha=alpha, adjust=False).mean().to_numpy()


def apply_clean_log_y_axis(ax, model):
    limits, ticks = Y_AXIS[model]
    ax.set_yscale("log")
    ax.set_ylim(*limits)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{tick:g}" for tick in ticks])
    ax.minorticks_off()


fig, axes = plt.subplots(4, 3, figsize=(10, 11), sharex=True, squeeze=False)
for row, model in enumerate(MODELS):
    values = []
    for col, run in enumerate(runs[runs.model == model].to_dict("records")):
        ax = axes[row, col]
        selected = tasks[(tasks.model == model) & (tasks.setting == run["setting"])]
        for probe in PROBES:
            curve = selected[selected.task == probe].sort_values(
                "cumulative_training_tokens"
            )
            x, y = curve.cumulative_training_tokens.to_numpy(float), curve.R_mean.to_numpy(float)
            keep = np.isfinite(x) & np.isfinite(y) & (x >= 0) & (y > 0)
            ax.plot(x[keep], ema(y[keep]), color=COLORS[probe], label=PROBE_LABELS[probe],
                    linewidth=1.9, marker="o", markersize=2.8)
            values.extend(y[keep])


        ax.set_title(run["setting"], fontsize=10, pad=7)
        apply_clean_log_y_axis(ax, model)
        ax.set_xlim(0, 300_000_000)
        ax.set_xticks([0, 100_000_000, 200_000_000, 300_000_000])
        ax.xaxis.set_major_formatter(FuncFormatter(millions))
        ax.grid(True, which="major", alpha=0.18, linewidth=0.55)
        ax.tick_params(axis="both", labelsize=8.5)

    axes[row, 0].set_ylabel(f"{MODEL_LABELS[model]}\n$R_{{\\mathcal{{D}}}}$", fontsize=10)

for ax in axes[-1]:
    ax.set_xlabel("Long trained Tokens", fontsize=10)

handles = [
    Line2D([0], [0], color=COLORS[p], marker="o", markersize=3, linewidth=1.9,
           label=PROBE_LABELS[p]) for p in PROBES
]
fig.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 1.01),
           ncol=6, frameon=False, fontsize=9, handlelength=2.2, columnspacing=1.2)
fig.subplots_adjust(left=0.09, right=0.995, bottom=0.065, top=0.92,
                    wspace=0.18, hspace=0.34)

output = OUT / "cpt_300m_R_by_model_and_setting.pdf"
fig.savefig(output, bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved: {output}")


In [ ]:
# 100M long-trained weight-decay comparison: WD=0 versus WD=0.1.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wandb
from matplotlib.lines import Line2D
from matplotlib.ticker import FuncFormatter, NullFormatter, NullLocator


ENTITY = "<WANDB_ENTITY>"
PROJECT = "plasticity-loss"
RUNS = {
    "WD = 0": "9cp4n3dg",
    "WD = 0.1": "nu93zzm9",
}
COLORS = {
    "WD = 0": "#4C78A8",
    "WD = 0.1": "#E45756",
}
DATASETS = ["gsm8k", "mbpp", "dolly_qa", "bio"]
LABELS = {
    "gsm8k": "GSM8K",
    "mbpp": "MBPP",
    "dolly_qa": "Dolly-QA",
    "bio": "PubMed (ID)",
}
R_KEYS = {
    dataset: f"plasticity/{dataset}/R_mean"
    for dataset in DATASETS
}
Y_AXIS = {
    "gsm8k": ((2.0, 6.0), [2, 3, 4, 5, 6]),
    "mbpp": ((0.75, 2.5), [0.8, 1, 1.5, 2, 2.5]),
    "dolly_qa": ((0.9, 3.0), [0.9, 1, 1.5, 2, 2.5, 3]),
    "bio": ((0.24, 0.6), [0.25, 0.3, 0.4, 0.5, 0.6]),
}
EMA_ALPHA = 0.18


def million_tokens(value, _):
    return f"{value / 1e6:g}M"


def ema(values, alpha=EMA_ALPHA):
    return pd.Series(values, dtype=float).ewm(
        alpha=alpha,
        adjust=False,
    ).mean().to_numpy()


def set_clean_log_y_axis(ax, limits, ticks):
    ax.set_yscale("log")
    ax.set_ylim(*limits)
    ax.set_yticks(ticks)
    ax.set_yticklabels([f"{tick:g}" for tick in ticks])
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_formatter(NullFormatter())


api = wandb.Api()
curves = {}

for label, run_id in RUNS.items():
    run = api.run(f"{ENTITY}/{PROJECT}/{run_id}")
    history = run.history(
        keys=["cumulative_training_tokens", *R_KEYS.values()],
        samples=10_000,
        pandas=True,
    )

    curves[label] = {}
    for dataset, metric in R_KEYS.items():
        curves[label][dataset] = (
            history[["cumulative_training_tokens", metric]]
            .dropna()
            .rename(
                columns={
                    "cumulative_training_tokens": "x",
                    metric: "R",
                }
            )
            .sort_values("x")
            .reset_index(drop=True)
        )

    print(f"{label}: {run.name} ({run.id})")


fig, axes = plt.subplots(
    1,
    len(DATASETS),
    figsize=(14.0, 3.45),
    sharex=True,
)

for ax, dataset in zip(axes, DATASETS):
    for label in RUNS:
        curve = curves[label][dataset]
        x = curve["x"].to_numpy(dtype=float)
        y = curve["R"].to_numpy(dtype=float)
        valid = np.isfinite(x) & np.isfinite(y) & (x >= 0) & (y > 0)
        x, y = x[valid], y[valid]

        # Faint raw trajectory plus the EMA used for visual comparison.
        ax.plot(
            x,
            y,
            color=COLORS[label],
            alpha=0.14,
            linewidth=0.65,
        )
        ax.plot(
            x,
            ema(y),
            color=COLORS[label],
            linewidth=2.0,
            label=label,
        )

    limits, ticks = Y_AXIS[dataset]
    set_clean_log_y_axis(ax, limits, ticks)
    ax.set_xlim(0, 100_000_000)
    ax.set_xticks([0, 25_000_000, 50_000_000, 75_000_000, 100_000_000])
    ax.xaxis.set_major_formatter(FuncFormatter(million_tokens))
    ax.set_title(LABELS[dataset], fontsize=12, pad=8)
    ax.set_xlabel("Long trained Tokens", fontsize=10)
    ax.grid(True, which="major", alpha=0.18, linewidth=0.6)
    ax.tick_params(axis="both", labelsize=8.5)

axes[0].set_ylabel(r"$R_{\mathcal{D}}$ (log scale)", fontsize=11)

handles = [
    Line2D([0], [0], color=COLORS[label], linewidth=2.1, label=label)
    for label in RUNS
]
axes[0].legend(
    handles=handles,
    loc="upper right",
    frameon=False,
    fontsize=9,
)

fig.subplots_adjust(
    left=0.065,
    right=0.995,
    bottom=0.18,
    top=0.80,
    wspace=0.22,
)

output_path = Path("./figures/cpt_100m_weight_decay_R.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(output_path, bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved: {output_path}")